# 04 — RAG Pipeline & Evaluation

Full end-to-end RAG pipeline with:
1. Retrieval + LLM generation
2. RAG vs. LLM-only comparison
3. Retrieval quality evaluation
4. Answer quality evaluation (LLM-as-judge)

### Key concepts
- **Grounding**: The LLM answers based on retrieved context, not its training data.
- **Faithfulness**: Does the answer only use information from the context?
- **LLM-as-judge**: Using an LLM to score the quality of another LLM's answers.

In [ ]:
import sys
sys.path.insert(0, "..")

import re
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv("../.env")

In [ ]:
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)
client = chromadb.PersistentClient(path="../chroma_db")
collection = client.get_collection(
    name="earnings_calls", embedding_function=embedding_fn,
)
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

print(f"Collection: {collection.count()} documents")

## RAG chain

The core pipeline: retrieve relevant chunks from ChromaDB, inject them as context into a prompt, and let the LLM generate a grounded answer. The prompt instructs the LLM to use ONLY the provided context and cite sources — this is what prevents hallucination.

In [ ]:
def retrieve(query, n_results=5, where=None):
    results = collection.query(query_texts=[query], n_results=n_results, where=where)
    context_parts = []
    for i in range(len(results["documents"][0])):
        doc = results["documents"][0][i]
        meta = results["metadatas"][0][i]
        context_parts.append(
            f"[{meta.get('speaker', '?')} — {meta.get('role', '?')} | {meta.get('quarter', '?')}]\n{doc}"
        )
    return results, "\n\n---\n\n".join(context_parts)

RAG_PROMPT = ChatPromptTemplate.from_template("""
You are an analyst assistant answering questions about earnings calls.
Use ONLY the provided context. If the answer is not in the context, say so.
Cite the speaker and quarter when relevant.

Context:
{context}

Question: {question}
""")

chain = RAG_PROMPT | llm | StrOutputParser()

def ask(question, n_results=5, where=None):
    results, context = retrieve(question, n_results, where)
    answer = chain.invoke({"context": context, "question": question})
    return results, context, answer

## Test queries

Verify the pipeline works with different types of questions: factual, speaker-specific, and off-topic. The off-topic query is important — a well-behaved RAG system should admit when the answer isn't in the context instead of making something up.

In [ ]:
q = "What was Apple's total revenue in Q4 2025?"
_, _, answer = ask(q)
print(f"Q: {q}\nA: {answer}")

In [ ]:
q = "What did the CFO say about gross margins?"
_, _, answer = ask(q)
print(f"Q: {q}\nA: {answer}")

In [ ]:
# Off-topic: should acknowledge the info isn't available
q = "What is Apple's strategy for electric vehicles?"
_, _, answer = ask(q)
print(f"Q: {q}\nA: {answer}")

## RAG vs. LLM-only

The whole point of RAG: the LLM with retrieved context gives specific, verifiable answers. Without context, it relies on training data which may be outdated, vague, or wrong for recent earnings calls.

In [ ]:
q = "What was Apple's gross margin in Q4 2025 and what drove it?"

_, _, rag_answer = ask(q)
llm_answer = llm.invoke(f"Answer about Apple's Q4 2025 earnings: {q}").content

print("WITH RAG:")
print(rag_answer)
print("\nWITHOUT RAG (LLM only):")
print(llm_answer)
print("\nThe RAG answer cites specific numbers from the transcript.")
print("The LLM-only answer may be vague or hallucinated.")

---
## Evaluation

A RAG system can fail in two places: the **retriever** doesn't find the right chunks, or the **generator** doesn't use them properly. We evaluate both separately.

### Test set

Each test case specifies: the question, which quarter/speaker should appear in results, and keywords that a correct answer must contain. This lets us measure retrieval and generation quality independently.

In [ ]:
test_cases = [
    {
        "question": "What were Apple's total revenue results in Q4 2025?",
        "expected_quarter": ["Q4-2025"],
        "expected_speaker": "Kevan Parekh",
        "expected_keywords": ["102.5", "billion", "8%"],
        "category": "factual",
    },
    {
        "question": "What was Apple's revenue in Q3 2025?",
        "expected_quarter": ["Q3-2025"],
        "expected_speaker": "Kevan Parekh",
        "expected_keywords": ["94", "billion", "10%"],
        "category": "factual",
    },
    {
        "question": "What did the CFO say about gross margins in Q4 2025?",
        "expected_quarter": ["Q4-2025"],
        "expected_speaker": "Kevan Parekh",
        "expected_keywords": ["margin", "gross"],
        "category": "speaker-specific",
    },
    {
        "question": "What did Tim Cook say about Apple Intelligence?",
        "expected_quarter": ["Q3-2025", "Q4-2025", "Q1-2026"],
        "expected_speaker": "Timothy D. Cook",
        "expected_keywords": ["Apple Intelligence"],
        "category": "speaker-specific",
    },
    {
        "question": "What were the iPhone results in Q3 2025?",
        "expected_quarter": ["Q3-2025"],
        "expected_speaker": "",
        "expected_keywords": ["iPhone"],
        "category": "quarter-specific",
    },
    {
        "question": "What is Apple's revenue guidance for the next quarter in Q1 2026?",
        "expected_quarter": ["Q1-2026"],
        "expected_speaker": "",
        "expected_keywords": ["guidance", "expect"],
        "category": "quarter-specific",
    },
    {
        "question": "How has Apple's revenue changed across quarters?",
        "expected_quarter": ["Q3-2025", "Q4-2025"],
        "expected_speaker": "",
        "expected_keywords": ["94", "102.5"],
        "category": "cross-quarter",
    },
    {
        "question": "What is Apple's strategy for electric vehicles?",
        "expected_quarter": [],
        "expected_speaker": "",
        "expected_keywords": [],
        "category": "off-topic",
    },
]

print(f"{len(test_cases)} test cases:")
for tc in test_cases:
    print(f"  [{tc['category']}] {tc['question'][:65]}")

### Retrieval evaluation

Checks whether the retriever brings back chunks from the right quarter, the right speaker, and containing the expected keywords. If retrieval fails here, the LLM has no chance of answering correctly.

In [ ]:
def evaluate_retrieval(test_cases):
    results_log = []
    for tc in test_cases:
        raw_results, context = retrieve(tc["question"])
        retrieved_quarters = [raw_results["metadatas"][0][i]["quarter"]
                             for i in range(len(raw_results["documents"][0]))]
        retrieved_speakers = [raw_results["metadatas"][0][i]["speaker"]
                             for i in range(len(raw_results["documents"][0]))]

        expected_q = tc.get("expected_quarter", [])
        expected_s = tc.get("expected_speaker", "")
        quarter_hit = any(q in retrieved_quarters for q in expected_q) if expected_q else True
        speaker_hit = expected_s in retrieved_speakers if expected_s else True

        context_lower = context.lower()
        keyword_hits = [kw for kw in tc["expected_keywords"] if kw.lower() in context_lower]
        keyword_score = len(keyword_hits) / len(tc["expected_keywords"]) if tc["expected_keywords"] else 1.0

        results_log.append({
            "question": tc["question"], "category": tc["category"],
            "quarter_hit": quarter_hit, "speaker_hit": speaker_hit,
            "keyword_score": keyword_score,
            "keyword_misses": [kw for kw in tc["expected_keywords"] if kw.lower() not in context_lower],
        })
    return results_log

retrieval_results = evaluate_retrieval(test_cases)

print("RETRIEVAL EVALUATION")
print("=" * 60)
for r in retrieval_results:
    status = "PASS" if r["quarter_hit"] and r["speaker_hit"] and r["keyword_score"] >= 0.5 else "FAIL"
    print(f"[{status}] [{r['category']}] {r['question'][:55]}")
    if r["keyword_misses"]:
        print(f"       Missing: {r['keyword_misses']}")

total = len(retrieval_results)
print(f"\nQuarter accuracy: {sum(r['quarter_hit'] for r in retrieval_results)}/{total}")
print(f"Speaker accuracy: {sum(r['speaker_hit'] for r in retrieval_results)}/{total}")
print(f"Avg keyword coverage: {sum(r['keyword_score'] for r in retrieval_results)/total:.0%}")

### Answer evaluation (LLM-as-judge)

We use the LLM itself to score generated answers on **faithfulness** (is it grounded in context?) and **relevance** (does it answer the question?). This is a common technique for evaluating RAG systems at scale when human evaluation is too expensive.

In [ ]:
JUDGE_PROMPT = ChatPromptTemplate.from_template("""
Evaluate this RAG system answer on two dimensions:

1. **Faithfulness** (1-5): Does the answer only use information from the context?
2. **Relevance** (1-5): Does the answer address the question?

Context:
{context}

Question: {question}
Answer: {answer}
Expected keywords: {expected_keywords}

Respond exactly as:
Faithfulness: [1-5]
Relevance: [1-5]
Brief explanation: [one sentence]
""")

judge_chain = JUDGE_PROMPT | llm | StrOutputParser()

In [ ]:
answer_results = []

for tc in test_cases:
    print(f"  {tc['question'][:50]}...", end=" ")
    raw_results, context, answer = ask(tc["question"])
    judgment = judge_chain.invoke({
        "context": context, "question": tc["question"],
        "answer": answer, "expected_keywords": ", ".join(tc["expected_keywords"]),
    })
    f_match = re.search(r"Faithfulness:\s*(\d)", judgment)
    r_match = re.search(r"Relevance:\s*(\d)", judgment)
    faithfulness = int(f_match.group(1)) if f_match else 0
    relevance = int(r_match.group(1)) if r_match else 0

    answer_results.append({
        "question": tc["question"], "category": tc["category"],
        "answer": answer, "faithfulness": faithfulness,
        "relevance": relevance, "judgment": judgment,
    })
    print(f"F:{faithfulness}/5 R:{relevance}/5")

In [ ]:
# Results summary
total = len(answer_results)
avg_f = sum(r["faithfulness"] for r in answer_results) / total
avg_r = sum(r["relevance"] for r in answer_results) / total

print(f"ANSWER EVALUATION — Avg Faithfulness: {avg_f:.1f}/5 | Avg Relevance: {avg_r:.1f}/5")
print("=" * 60)

for r in answer_results:
    print(f"[{r['category']}] F:{r['faithfulness']} R:{r['relevance']} — {r['question'][:55]}")

# Per-category
print("\nPer category:")
for cat in sorted(set(r["category"] for r in answer_results)):
    cr = [r for r in answer_results if r["category"] == cat]
    print(f"  {cat}: F={sum(r['faithfulness'] for r in cr)/len(cr):.1f} R={sum(r['relevance'] for r in cr)/len(cr):.1f} (n={len(cr)})")

In [ ]:
# Weakest answers
print("WEAKEST ANSWERS:")
print("=" * 60)
for r in sorted(answer_results, key=lambda x: x["faithfulness"] + x["relevance"])[:3]:
    print(f"\n[{r['category']}] {r['question']}")
    print(f"  F:{r['faithfulness']}/5 R:{r['relevance']}/5")
    print(f"  Answer: {r['answer'][:200]}...")